# Sidekick

LangGraph worker, tool loop, evaluator, Gradio session state. Optional session context on every turn; extra tools: `fetch_url_text`, `append_worklog`, `pretty_json`, `list_sandbox_files`.

Run the notebook from this folder so `sandbox/` is created next to it. After the first pip cell, run once: `playwright install chromium`.

Env: `OPENAI_API_KEY`; `SERPER_API_KEY` for search; Pushover vars for push; `SIDEKICK_HEADLESS=1` for headless browser.


In [ ]:
%pip install -q gradio python-dotenv langgraph langchain-openai langchain-community langchain-experimental playwright requests pydantic typing_extensions


In [ ]:
import asyncio
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated, Any, Dict, List, Optional
from urllib.parse import urlparse

import gradio as gr
import requests
from dotenv import load_dotenv
from langchain.agents import Tool
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_experimental.tools import PythonREPLTool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from playwright.async_api import async_playwright
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv(override=True)

SANDBOX_DIR = (Path.cwd() / "sandbox").resolve()
SANDBOX_DIR.mkdir(parents=True, exist_ok=True)

pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"
_serper_client = None


In [ ]:
def _headless() -> bool:
    return os.getenv("SIDEKICK_HEADLESS", "").strip().lower() in ("1", "true", "yes")


async def playwright_tools():
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=_headless())
    toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=browser)
    return toolkit.get_tools(), browser, playwright


def push(text: str) -> str:
    if not pushover_token or not pushover_user:
        return "Pushover not configured (set PUSHOVER_TOKEN and PUSHOVER_USER)."
    try:
        r = requests.post(
            pushover_url,
            data={"token": pushover_token, "user": pushover_user, "message": text},
            timeout=15,
        )
        r.raise_for_status()
    except requests.RequestException as e:
        return f"Push failed: {e}"
    return "ok"


def get_file_tools():
    toolkit = FileManagementToolkit(root_dir=str(SANDBOX_DIR))
    return toolkit.get_tools()


def http_get_text(url: str) -> str:
    limit = 12000
    parsed = urlparse(url.strip())
    if parsed.scheme not in ("http", "https") or not parsed.netloc:
        return "Only http/https URLs with a host are allowed."
    try:
        r = requests.get(
            url.strip(),
            timeout=20,
            headers={"User-Agent": "SidekickFetch/1.0"},
        )
    except requests.RequestException as e:
        return f"Request error: {e}"
    text = r.text or ""
    if len(text) > limit:
        text = text[:limit] + f"\n… truncated ({len(r.text)} chars total)"
    return f"status={r.status_code}\n\n{text}"


def worklog_append(line: str) -> str:
    entry = line.replace("\n", " ").replace("\r", "").strip()
    if not entry:
        return "Nothing to append."
    path = SANDBOX_DIR / "worklog.md"
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    block = f"- {ts} {entry}\n"
    with path.open("a", encoding="utf-8") as f:
        f.write(block)
    return f"Appended to {path.name}"


def format_json_string(payload: str) -> str:
    try:
        data = json.loads(payload)
    except json.JSONDecodeError as e:
        return f"Invalid JSON: {e}"
    return json.dumps(data, indent=2, ensure_ascii=False)


def list_sandbox_files(_unused: str = "") -> str:
    cap = 120
    paths: list[str] = []
    for root, dirs, files in os.walk(SANDBOX_DIR):
        dirs.sort()
        for name in sorted(files):
            full = Path(root) / name
            try:
                rel = full.relative_to(SANDBOX_DIR)
            except ValueError:
                continue
            paths.append(str(rel))
            if len(paths) >= cap:
                return "\n".join(paths) + f"\n… stopped at {cap} entries"
    return "\n".join(paths) if paths else "(empty sandbox)"


def _search_or_stub(query: str) -> str:
    global _serper_client
    if not os.getenv("SERPER_API_KEY"):
        return "Web search needs SERPER_API_KEY in the environment."
    if _serper_client is None:
        _serper_client = GoogleSerperAPIWrapper()
    return _serper_client.run(query)


async def other_tools():
    push_tool = Tool(
        name="send_push_notification",
        func=push,
        description="Send a short push notification via Pushover when the user explicitly wants an alert on their device.",
    )
    file_tools = get_file_tools()
    search_tool = Tool(
        name="search",
        func=_search_or_stub,
        description="Run a web search and return summarized results.",
    )
    wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    python_repl = PythonREPLTool()
    http_tool = Tool(
        name="fetch_url_text",
        func=http_get_text,
        description="GET a public http(s) URL and return status plus body text (truncated).",
    )
    log_tool = Tool(
        name="append_worklog",
        func=worklog_append,
        description="Append one timestamped bullet line to sandbox/worklog.md for milestones or decisions.",
    )
    json_tool = Tool(
        name="pretty_json",
        func=format_json_string,
        description="Parse a JSON string and return a formatted version; errors if invalid.",
    )
    ls_tool = Tool(
        name="list_sandbox_files",
        func=list_sandbox_files,
        description="List up to 120 files under the sandbox directory.",
    )
    return file_tools + [
        push_tool,
        search_tool,
        wiki,
        python_repl,
        http_tool,
        log_tool,
        json_tool,
        ls_tool,
    ]


In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(
        description="True if more input is needed from the user, or clarifications, or the assistant is stuck"
    )


class Sidekick:
    def __init__(self):
        self.worker_llm_with_tools = None
        self.evaluator_llm_with_output = None
        self.tools = None
        self.graph = None
        self.sidekick_id = str(uuid.uuid4())
        self.memory = MemorySaver()
        self.browser = None
        self.playwright = None

    async def setup(self):
        self.tools, self.browser, self.playwright = await playwright_tools()
        self.tools += await other_tools()
        worker_llm = ChatOpenAI(model="gpt-4o-mini")
        self.worker_llm_with_tools = worker_llm.bind_tools(self.tools)
        evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
        self.evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)
        await self.build_graph()

    def worker(self, state: State) -> Dict[str, Any]:
        system_message = f"""You are a capable assistant with tools: web search, optional browser navigation, Wikipedia, Python (use print() for output), sandbox file I/O, Pushover notifications when asked, direct HTTP fetch for a URL, pretty-print JSON, a running worklog in the sandbox, and a sandbox file listing.

When a request is underspecified (missing output shape, audience, geography, time window, or constraints), ask up to three numbered questions and wait—do not burn search or browser steps until those basics are clear unless the user insists on proceeding.

Keep answers aligned with the success criteria. If you need the user, start with "Question:" and be explicit.

Current local time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Success criteria:
{state["success_criteria"]}
"""

        if state.get("feedback_on_work"):
            system_message += f"""

Earlier attempt did not meet the bar. Evaluator feedback:
{state["feedback_on_work"]}
Address this and continue.
"""

        found_system_message = False
        messages = state["messages"]
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system_message = True

        if not found_system_message:
            messages = [SystemMessage(content=system_message)] + messages

        response = self.worker_llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def worker_router(self, state: State) -> str:
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools"
        return "evaluator"

    def format_conversation(self, messages: List[Any]) -> str:
        conversation = "Conversation history:\n\n"
        for message in messages:
            if isinstance(message, HumanMessage):
                conversation += f"User: {message.content}\n"
            elif isinstance(message, AIMessage):
                text = message.content or "[tool calls]"
                conversation += f"Assistant: {text}\n"
        return conversation

    def evaluator(self, state: State) -> State:
        last_response = state["messages"][-1].content

        system_message = (
            "You grade whether the assistant finished the job against the stated success criteria. "
            "Decide if criteria are met and whether the user still owes input."
        )

        user_message = f"""Conversation:
{self.format_conversation(state["messages"])}

Success criteria:
{state["success_criteria"]}

Last assistant message (not counting tool traces):
{last_response}

If the assistant claimed a file was written, trust tool use in the trace when visible; otherwise apply normal skepticism."""

        if state["feedback_on_work"]:
            user_message += f"\nPrior feedback was: {state['feedback_on_work']}\nIf the same failure repeats, mark user_input_needed."

        eval_result = self.evaluator_llm_with_output.invoke(
            [
                SystemMessage(content=system_message),
                HumanMessage(content=user_message),
            ]
        )
        return {
            "messages": [
                {
                    "role": "assistant",
                    "content": f"Evaluator Feedback on this answer: {eval_result.feedback}",
                }
            ],
            "feedback_on_work": eval_result.feedback,
            "success_criteria_met": eval_result.success_criteria_met,
            "user_input_needed": eval_result.user_input_needed,
        }

    def route_based_on_evaluation(self, state: State) -> str:
        if state["success_criteria_met"] or state["user_input_needed"]:
            return "END"
        return "worker"

    async def build_graph(self):
        graph_builder = StateGraph(State)
        graph_builder.add_node("worker", self.worker)
        graph_builder.add_node("tools", ToolNode(tools=self.tools))
        graph_builder.add_node("evaluator", self.evaluator)
        graph_builder.add_conditional_edges(
            "worker", self.worker_router, {"tools": "tools", "evaluator": "evaluator"}
        )
        graph_builder.add_edge("tools", "worker")
        graph_builder.add_conditional_edges(
            "evaluator", self.route_based_on_evaluation, {"worker": "worker", "END": END}
        )
        graph_builder.add_edge(START, "worker")
        self.graph = graph_builder.compile(checkpointer=self.memory)

    async def run_superstep(
        self,
        user_message: str,
        success_criteria: str,
        history: Optional[List[Any]],
        session_context: str = "",
    ):
        config = {"configurable": {"thread_id": self.sidekick_id}, "recursion_limit": 45}
        model_message = user_message.strip()
        ctx = (session_context or "").strip()
        if ctx:
            model_message = f"[Session context]\n{ctx}\n\n[Request]\n{user_message.strip()}"

        state = {
            "messages": model_message,
            "success_criteria": success_criteria or "The answer should be clear and accurate",
            "feedback_on_work": None,
            "success_criteria_met": False,
            "user_input_needed": False,
        }
        result = await self.graph.ainvoke(state, config=config)
        history = history or []
        user = {"role": "user", "content": user_message.strip()}
        reply = {"role": "assistant", "content": result["messages"][-2].content}
        feedback = {"role": "assistant", "content": result["messages"][-1].content}
        return history + [user, reply, feedback]

    def cleanup(self):
        if not self.browser:
            return
        try:
            loop = asyncio.get_running_loop()
            loop.create_task(self.browser.close())
            if self.playwright:
                loop.create_task(self.playwright.stop())
        except RuntimeError:
            asyncio.run(self.browser.close())
            if self.playwright:
                asyncio.run(self.playwright.stop())


In [ ]:
async def setup():
    sk = Sidekick()
    await sk.setup()
    return sk


async def process_message(sidekick, session_context, message, success_criteria, history):
    if history is None:
        history = []
    results = await sidekick.run_superstep(
        message,
        success_criteria,
        history,
        session_context=session_context or "",
    )
    return results, sidekick


async def reset():
    sk = Sidekick()
    await sk.setup()
    return "", "", "", None, sk


def free_resources(sidekick):
    try:
        if sidekick:
            sidekick.cleanup()
    except Exception as e:
        print(f"cleanup: {e}")


with gr.Blocks(title="Sidekick", theme=gr.themes.Default(primary_hue="teal")) as ui:
    gr.Markdown("## Sidekick")
    sidekick_state = gr.State(delete_callback=free_resources)

    with gr.Row():
        chatbot = gr.Chatbot(label="Chat", height=360, type="messages")
    session_context = gr.Textbox(
        label="Session context",
        placeholder="Optional: timezone, stack, tone, or constraints that apply to every turn.",
        lines=2,
    )
    message = gr.Textbox(label="Request", placeholder="What should get done?")
    success_criteria = gr.Textbox(
        label="Success criteria",
        placeholder="How you will judge completion.",
    )
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go", variant="primary")

    ui.load(setup, [], [sidekick_state])
    go_button.click(
        process_message,
        [sidekick_state, session_context, message, success_criteria, chatbot],
        [chatbot, sidekick_state],
    )
    message.submit(
        process_message,
        [sidekick_state, session_context, message, success_criteria, chatbot],
        [chatbot, sidekick_state],
    )
    success_criteria.submit(
        process_message,
        [sidekick_state, session_context, message, success_criteria, chatbot],
        [chatbot, sidekick_state],
    )
    reset_button.click(
        reset, [], [session_context, message, success_criteria, chatbot, sidekick_state]
    )

ui.launch(inbrowser=True)
